In [3]:
# Fix NumPy compatibility issue
!pip install numpy==1.26.4 --force-reinstall
!pip install pandas --upgrade
!pip install wfdb --upgrade

  Obtaining dependency information for numpy==1.26.4 from https://files.pythonhosted.org/packages/3f/6b/5610004206cf7f8e7ad91c5a85a8c71b2f2f8051a0c0c4d5916b76d6cbb2/numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata
     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ------------ ------------------------- 20.5/61.0 kB 330.3 kB/s eta 0:00:01
     ------------------------------- ------ 51.2/61.0 kB 435.7 kB/s eta 0:00:01
     -------------------------------------- 61.0/61.0 kB 466.1 kB/s eta 0:00:00
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.1/15.8 MB 1.7 MB/s eta 0:00:10
   ---------------------------------------- 0.2/15.8 MB 2.6 MB/s eta 0:00:06
    --------------------------------------- 0.3/15.8 MB 2.4 MB/s eta 0:00:07
   - -------------------------------------- 0.5/15.8 MB 2.7 MB/s eta 0:00:06
   - -------------------------------------- 0.6/15.8 MB 2.8 MB/s eta 0:00:06
   - ---------------

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\ASUS\\anaconda3\\Lib\\site-packages\\~umpy.libs\\libscipy_openblas64_-860d95b1c38e637ce4509f5fa24fbf2a.dll'
Consider using the `--user` option or check the permissions.



  Obtaining dependency information for pandas from https://files.pythonhosted.org/packages/eb/62/c321f13b5ba1819fc8dca456c7fce578da2dcfecff1abbf0eaddf8406c0f/pandas-3.0.3-cp311-cp311-win_amd64.whl.metadata
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB 960.0 kB/s eta 0:00:11
   ---------------------------------------- 0.1/9.9 MB 1.4 MB/s eta 0:00:07
    --------------------------------------- 0.2/9.9 MB 1.5 MB/s eta 0:00:07
   - -------------------------------------- 0.4/9.9 MB 2.1 MB/s eta 0:00:05
   - -------------------------------------- 0.4/9.9 MB 2.1 MB/s eta 0:00:05
   - -------------------------------------- 0.4/9.9 MB 2.1 MB/s eta 0:00:05
   - -------------------------------------- 0.4/9.9 MB 2.1 MB/s eta 0:00:05
   - -------------------------------------- 0.4/9.9 MB 1.1 MB/s eta 0:00:09
   - -------------------------------------- 0.5/9.9 MB 1.1 MB/s eta 0:00:09
   -- ---------------------------------

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\ASUS\\anaconda3\\Lib\\site-packages\\~andas\\_libs\\algos.cp311-win_amd64.pyd'
Consider using the `--user` option or check the permissions.



  Obtaining dependency information for wfdb from https://files.pythonhosted.org/packages/d7/45/05cc2ecbf61163bbbb18678dcdc7e7e7285f9f50f4dcf96a9ff07633ac52/wfdb-4.3.1-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/163.9 kB ? eta -:--:--
   -- ------------------------------------- 10.2/163.9 kB ? eta -:--:--
   -- ------------------------------------- 10.2/163.9 kB ? eta -:--:--
   -------------- ------------------------ 61.4/163.9 kB 544.7 kB/s eta 0:00:01
   -------------------------- ----------- 112.6/163.9 kB 819.2 kB/s eta 0:00:01
   --------------------------------- ---- 143.4/163.9 kB 708.1 kB/s eta 0:00:01
   -------------------------------------- 163.9/163.9 kB 701.8 kB/s eta 0:00:00
  Attempting uninstall: wfdb
    Found existing installation: wfdb 4.3.0
    Uninstalling wfdb-4.3.0:
      Successfully uninstalled wfdb-4.3.0


In [1]:
# ================================================
# FINAL WORKING CODE - Simplified wrsamp
# ================================================

import wfdb
import numpy as np
from scipy.signal import resample
import os
from tqdm import tqdm

# ====================== CONFIGURATION ======================
input_dir = r"E:\ECG_MIT-BIH_SIN_Holter\mit-bih-normal-sinus-rhythm-database-1.0.0"
output_dir = r"E:\ECG_MIT-BIH_SIN_Holter\nsrdb_250hz"

os.makedirs(output_dir, exist_ok=True)

print(f"Input : {input_dir}")
print(f"Output: {output_dir}")
print("-" * 60)

record_names = [f[:-4] for f in os.listdir(input_dir) if f.endswith('.hea')]
print(f"Found {len(record_names)} records.\n")

successful = 0
failed = 0
original_dir = os.getcwd()

for record_name in tqdm(record_names, desc="Upsampling"):
    try:
        # Read record
        os.chdir(input_dir)
        record = wfdb.rdrecord(record_name)
        
        # Upsampling
        upsampled_signals = []
        for ch in record.p_signal.T:
            new_length = int(len(ch) * (250.0 / 128.0))
            upsampled_ch = resample(ch, new_length)
            upsampled_signals.append(upsampled_ch)
        
        upsampled_signals = np.array(upsampled_signals).T
        
        # Save - Simplified call
        os.chdir(output_dir)
        
        wfdb.wrsamp(
            record_name=record_name,
            fs=250,
            sig_name=record.sig_name,
            p_signal=upsampled_signals,
            units=record.units,
            fmt=['16' for _ in record.sig_name],   # Important fix
            adc_gain=[1000 for _ in record.sig_name],  # Standard value
            baseline=[0 for _ in record.sig_name]
        )
        
        successful += 1
        print(f"  ✅ {record_name} saved")
        
    except Exception as e:
        print(f"  ❌ Failed {record_name}: {e}")
        failed += 1
    
    finally:
        os.chdir(original_dir)

print("\n" + "="*80)
print(f"🎉 UPSAMPLING COMPLETED!")
print(f"   Successful : {successful}/18")
print(f"   Failed     : {failed}")
print(f"   Output folder: {output_dir}")
print("="*80)

C:\Users\ASUS\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\ASUS\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Input : E:\ECG_MIT-BIH_SIN_Holter\mit-bih-normal-sinus-rhythm-database-1.0.0
Output: E:\ECG_MIT-BIH_SIN_Holter\nsrdb_250hz
------------------------------------------------------------
Found 18 records.



Upsampling:   6%|███▉                                                                   | 1/18 [00:13<03:56, 13.91s/it]

  ✅ 16265 saved


Upsampling:  11%|███████▉                                                               | 2/18 [00:24<03:10, 11.88s/it]

...

  ✅ 19093 saved


Upsampling:  94%|██████████████████████████████████████████████████████████████████    | 17/18 [04:04<00:14, 14.23s/it]

  ✅ 19140 saved


Upsampling: 100%|██████████████████████████████████████████████████████████████████████| 18/18 [04:33<00:00, 15.18s/it]

  ✅ 19830 saved

🎉 UPSAMPLING COMPLETED!
   Successful : 18/18
   Failed     : 0
   Output folder: E:\ECG_MIT-BIH_SIN_Holter\nsrdb_250hz


In [2]:
import wfdb
print(wfdb.__version__)

4.3.1


In [2]:
import os

output_dir = r"E:\ECG_MIT-BIH_SIN_Holter\nsrdb_250hz"
files = os.listdir(output_dir)

hea_files = [f for f in files if f.endswith('.hea')]
dat_files = [f for f in files if f.endswith('.dat')]

print(f"✅ Total .hea files: {len(hea_files)}")
print(f"✅ Total .dat files: {len(dat_files)}")
print(f"Should be 18 each.")

✅ Total .hea files: 18
✅ Total .dat files: 18
Should be 18 each.


In [4]:
import wfdb
import os

output_dir = r"E:\ECG_MIT-BIH_SIN_Holter\nsrdb_250hz"
original_dir = os.getcwd()

sample_records = ['16265', '16272', '19830']

print("=== CHECKING UPSAMPLED RECORDS ===\n")

for rec in sample_records:
    try:
        os.chdir(output_dir)
        record = wfdb.rdrecord(rec)   # No base_dir
        
        print(f"✅ {rec:6} | Sampling Rate: {record.fs} Hz | "
              f"Channels: {record.n_sig} | "
              f"Duration: {len(record.p_signal)/record.fs:.1f} seconds")
        
    except Exception as e:
        print(f"❌ Error reading {rec}: {e}")
    finally:
        os.chdir(original_dir)

print("\n" + "="*60)

=== CHECKING UPSAMPLED RECORDS ===

✅ 16265  | Sampling Rate: 250 Hz | Channels: 2 | Duration: 91648.0 seconds
✅ 16272  | Sampling Rate: 250 Hz | Channels: 2 | Duration: 90000.0 seconds
✅ 19830  | Sampling Rate: 250 Hz | Channels: 2 | Duration: 83608.0 seconds



In [5]:
import shutil

old_folder = r"E:\ECG_MIT-BIH_SIN_Holter\mit-bih-normal-sinus-rhythm-database-1.0.0"
shutil.rmtree(old_folder)
print("✅ Old folder deleted.")

✅ Old folder deleted.


In [1]:
import os
print(os.getcwd())


C:\Users\ASUS
